# Lab 1.1 · Triggered and diluted effects in SQL

Day 1 · Session 1 (Assignment, exposure and triggering) · 25 min.

Sufrah randomised all 200,000 active customers into test T-101, AI-ranked search, on day 0. Only
customers who searched ever saw the new ranking. You trace assignment and exposure in SQL over the event
stream, then estimate the effect twice: over everyone assigned (diluted) and over the triggered customers.

**AI policy: AI-tutor.** Write the SQL yourself; an assistant may explain a join or a window function.

**Reference state.** The T-101 event stream, generated by `sufrah.py` with a fixed seed. Nothing from another lab.

**Tasks**
- [ ] 1 · Read the four tables and their keys (4')
- [ ] 2 · Join assignments to the first search: who was exposed, when (6')
- [ ] 3 · Orders per customer, all assigned against triggered, in SQL (6')
- [ ] 4 · Two effects with intervals, and the dilution formula (6')
- [ ] 5 · Check: exposure is balanced between the arms (3')
- [ ] ✱ Stretch · Count only orders after exposure and say what changes

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/SDAIAAcademy/ds-302.git"
    if not Path("/content/online-experiments").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/online-experiments"], check=True)
    os.chdir("/content/online-experiments/labs")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    while not Path("sufrah.py").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")  # solutions/ notebooks run from labs/
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
from scipy import stats
import sufrah as U

S = U.stream()
print("mode:", U.MODE, "| customers:", len(S["customers"]), "| orders:", len(S["orders"]))

## 1 · The four tables (4')

| Table | One row per | Keys |
|---|---|---|
| `customers` | customer | `customer_id`, `city`, `platform` |
| `assignments` | customer in T-101 | `customer_id`, `variant`, `assigned_day` (0) |
| `searches` | search | `customer_id`, `day`, `latency_ms`, `cost_sar` |
| `orders` | order | `customer_id`, `day` (negative in the 28 pre-period days), `value`, `complaint` |

✏️ **Exercise** · Write `count_sql()`: a query that returns one row per variant with the number of customers
assigned.

In [ ]:
def count_sql() -> str:
    return "select variant, count(*) as customers from assignments group by variant order by variant"

In [ ]:
# Check, with a safety net: the cells below still run when the exercise is not done yet
try:
    q = count_sql()
except NotImplementedError:
    q = U.reference("d1_1_count_sql")
U.keep("d1_1_count_sql", q)
counts = U.sql(q)
print(counts)
U.check(int(counts["customers"].sum()) == len(S["customers"]), "every customer is assigned once")

## 2 · Who was exposed, and when (6')

Assignment happens on day 0 for everyone. **Exposure** happens at a customer's first search in the test
period (day 0 or later): that is when the ranking could first make a difference. The **trigger condition** is
"searched at least once"; it is logged in both arms, so control customers who *would* have seen the new
ranking are flagged too (counterfactual logging).

✏️ **Exercise** · Write `exposure_sql()`: one row per assigned customer with `customer_id`, `variant`, and
`exposed_day` (the first search day, or NULL when the customer never searched). Use a left join.

In [ ]:
def exposure_sql() -> str:
    return """
    select a.customer_id, a.variant, f.exposed_day
    from assignments a
    left join (select customer_id, min(day) as exposed_day
               from searches where day >= 0 group by customer_id) f
      on f.customer_id = a.customer_id
    """

In [ ]:
try:
    q = exposure_sql()
except NotImplementedError:
    q = U.reference("d1_1_exposure_sql")
U.keep("d1_1_exposure_sql", q)
exposure = U.sql(q)
rate = exposure.exposed_day.notna().mean()
print(f"triggered: {rate:.1%} of assigned customers")
print(exposure.groupby("variant").exposed_day.describe()[["count", "mean", "50%"]])
U.check(len(exposure) == len(S["customers"]) and 0.3 < rate < 0.6, "one row per assigned customer; 30 to 60% triggered")

## 3 · Orders per customer, in SQL (6')

✏️ **Exercise** · Write `per_customer_sql()`: one row per assigned customer with `customer_id`, `variant`,
`triggered` (true or false) and `orders` (orders in the test period, day 0 to 27; zero when none). Customers
without an order must stay in the table with 0: dropping them is the most common bug in this query.

In [ ]:
def per_customer_sql() -> str:
    return """
    with first_search as (
        select customer_id, min(day) as exposed_day from searches where day >= 0 group by customer_id),
    test_orders as (
        select customer_id, count(*) as orders from orders where day >= 0 group by customer_id)
    select a.customer_id, a.variant,
           f.exposed_day is not null as triggered,
           coalesce(o.orders, 0) as orders
    from assignments a
    left join first_search f using (customer_id)
    left join test_orders o using (customer_id)
    """

In [ ]:
try:
    q = per_customer_sql()
except NotImplementedError:
    q = U.reference("d1_1_per_customer_sql")
U.keep("d1_1_per_customer_sql", q)
pc = U.sql(q)
print(pc.groupby(["triggered", "variant"]).orders.agg(["size", "mean"]).round(3))
U.check(len(pc) == len(S["customers"]) and (pc.orders == 0).any(), "every assigned customer, zeros kept")

## 4 · Two effects, with intervals (6')

✏️ **Exercise** · Write `diff_ci(df, metric)`: the difference in means treatment minus control, its 95% interval
(normal approximation, unequal variances) and the relative difference. Return a dict with `diff`, `lo`, `hi`,
`rel`, `se`.

In [ ]:
def diff_ci(df: pd.DataFrame, metric: str = "orders") -> dict:
    t = df.loc[df.variant == "treatment", metric]
    c = df.loc[df.variant == "control", metric]
    d = t.mean() - c.mean()
    se = np.sqrt(t.var(ddof=1) / len(t) + c.var(ddof=1) / len(c))
    return {"diff": d, "se": se, "lo": d - 1.96 * se, "hi": d + 1.96 * se, "rel": d / c.mean()}

In [ ]:
try:
    overall, triggered = diff_ci(pc), diff_ci(pc[pc.triggered])
except NotImplementedError:
    overall, triggered = U.reference("d1_1_overall"), U.reference("d1_1_triggered")
U.keep("d1_1_overall", overall)
U.keep("d1_1_triggered", triggered)
show = pd.DataFrame({"all assigned (diluted)": overall, "triggered": triggered}).T
print(show[["diff", "lo", "hi", "rel"]].round(4))
dilution = triggered["diff"] * rate
print(f"triggered difference x trigger rate = {dilution:.4f}   (overall difference {overall['diff']:.4f})")
U.check(abs(dilution - overall["diff"]) < 2 * overall["se"], "the overall effect is the triggered effect, diluted")
U.check(triggered["diff"] / triggered["se"] > overall["diff"] / overall["se"],
        "the triggered estimate stands more standard errors away from zero")

🔍 **Look at** the two `rel` values. The diluted relative lift is smaller because more than half of the customers
could not be affected at all, and their noise is still in the average. A decision about the ranking concerns
the people who search; report the triggered effect, and give the diluted one for the whole-business view.

## 5 · Is exposure balanced? (3')

Triggering is valid only if the treatment cannot change who triggers. Here the trigger (a search) happens
before the ranking is shown, so the trigger rate must be equal in both arms.

✏️ **Exercise** · Compute `balance_p`: the p-value of a chi-square test of triggered against variant.

In [ ]:
def trigger_balance(df: pd.DataFrame) -> float:
    table = pd.crosstab(df.variant, df.triggered)
    return float(stats.chi2_contingency(table)[1])

In [ ]:
try:
    balance_p = trigger_balance(pc)
except NotImplementedError:
    balance_p = U.reference("d1_1_balance_p")
U.keep("d1_1_balance_p", balance_p)
print(pd.crosstab(pc.variant, pc.triggered, normalize="index").round(4), "\np =", round(balance_p, 3))
U.check(balance_p > 0.001, "trigger rates do not differ between the arms")

## ✱ Stretch · Orders after exposure only

Count only orders on or after each customer's exposure day. The difference does not change much, but the
control mean falls: why? Which version would you pre-register, and why must it be computed the same way in
both arms?

## Checkpoint

1. A test randomised 100,000 customers; 30,000 triggered. The triggered lift is 4%. Roughly what lift do you
   expect over everyone assigned? <details><summary>Answer</summary>About 1.2% (4% x 0.3), if the triggered and
   untriggered customers have similar baselines; otherwise weight by their baselines.</details>
2. Why must control customers who searched be flagged too? <details><summary>Answer</summary>Without the flag in
   control you would compare triggered treatment customers with all control customers: different people.
   </details>
3. When is a trigger condition invalid? <details><summary>Answer</summary>When the treatment can change who
   triggers, for example a trigger defined by an event that the new feature itself causes.</details>

**Carry forward.** Lab 1.4 sizes this test on triggered customers; Day 2 analyses its results.